# save.js → 函数式 notebook

原 `class SaveSystem` 的全部 **37 个静态方法**拆成同名顶层函数：
`this.xxx()` → `xxx()`，`static MAX_SLOTS` → `const MAX_SLOTS`。

方法清单：`getUserList`、`_saveUserList`、`userExists`、`_getPasswordKey`、`setPassword`、`checkPassword`、`hasPassword`、`createUser`、`getDeviceUser`、`bindDeviceUser`、`deleteUser`、`getSaveKey`、`getSlotSaveKey`、`saveSlot`、`loadSlot`、`hasSlotSave`、`deleteSlotSave`、`getSlotInfo`、`getUsedSlotCount`、`applySlotSave`、`getMpSaveKey`、`_serializeBuildings`、`saveMultiplayer`、`loadMultiplayer`、`hasMpSave`、`applyMpSave`、`_applyData`、`save`、`load`、`hasSave`、`getSaveInfo`、`applySave`、`getArcadeSaveKey`、`saveArcade`、`hasArcadeSave`、`loadArcadeSave`、`applyArcadeSave`

In [ ]:
// ===== 共享常量（原 static 字段）=====
const MAX_SLOTS = 5; // 存档槽位上限（最多5个）

In [ ]:
// ===== SaveSystem.getUserList =====
function getUserList() {
        try {
            return JSON.parse(localStorage.getItem('ws_users') || '[]');
        } catch { return []; }
    }

In [ ]:
// ===== SaveSystem._saveUserList =====
function _saveUserList(list) {
        localStorage.setItem('ws_users', JSON.stringify(list));
    }

In [ ]:
// ===== SaveSystem.userExists =====
function userExists(username) {
        return getUserList().includes(username);
    }

In [ ]:
// ===== SaveSystem._getPasswordKey =====
function _getPasswordKey(username) {
        return 'ws_pass_' + username;
    }

In [ ]:
// ===== SaveSystem.setPassword =====
function setPassword(username, password) {
        localStorage.setItem(_getPasswordKey(username), password);
    }

In [ ]:
// ===== SaveSystem.checkPassword =====
function checkPassword(username, password) {
        const stored = localStorage.getItem(_getPasswordKey(username));
        return stored === password;
    }

In [ ]:
// ===== SaveSystem.hasPassword =====
function hasPassword(username) {
        return localStorage.getItem(_getPasswordKey(username)) !== null;
    }

In [ ]:
// ===== SaveSystem.createUser =====
function createUser(username, password) {
        const list = getUserList();
        if (!list.includes(username)) {
            list.push(username);
            _saveUserList(list);
        }
        if (password) setPassword(username, password);
    }

In [ ]:
// ===== SaveSystem.getDeviceUser =====
function getDeviceUser() {
        return localStorage.getItem('ws_device_user');
    }

In [ ]:
// ===== SaveSystem.bindDeviceUser =====
function bindDeviceUser(username) {
        localStorage.setItem('ws_device_user', username);
    }

In [ ]:
// ===== SaveSystem.deleteUser =====
function deleteUser(username) {
        const list = getUserList().filter(u => u !== username);
        _saveUserList(list);
        localStorage.removeItem('ws_save_' + username);
        for (let i = 1; i <= MAX_SLOTS; i++) {
            localStorage.removeItem(getSlotSaveKey(username, i));
        }
        localStorage.removeItem(_getPasswordKey(username));
        // 删除的是本机绑定账号时解除绑定，避免机器被锁死
        if (getDeviceUser() === username) {
            localStorage.removeItem('ws_device_user');
        }
    }

In [ ]:
// ===== SaveSystem.getSaveKey =====
function getSaveKey(username) {
        return 'ws_save_' + username;
    }

In [ ]:
// ===== SaveSystem.getSlotSaveKey =====
function getSlotSaveKey(username, slot) {
        return 'ws_save_' + username + '_slot' + slot;
    }

In [ ]:
// ===== SaveSystem.saveSlot =====
function saveSlot(username, slot, game) {
        if (!username || !game) return;
        const normalBldgs = game.world.worldType === 'normal' ? game.world.buildings : (game.world._normalBuildings || []);
        const stoneBldgs = game.world.worldType === 'stone' ? game.world.buildings : (game.world._stoneBuildings || []);
        const goldBldgs = game.world.worldType === 'gold' ? game.world.buildings : (game.world._goldBuildings || []);
        const data = {
            version: 1,
            timestamp: Date.now(),
            player: {
                x: game.player.position.x,
                y: game.player.position.y,
                z: game.player.position.z,
                yaw: game.player.yaw,
                pitch: game.player.pitch,
            },
            inventory: game.inventory.items.map(item => item ? { id: item.id, amount: item.amount, charge: item.charge || 0, uses: item.uses || 0 } : null),
            equipment: { ...game.inventory.equipment },
            survival: {
                health: game.survival.health,
                hunger: game.survival.hunger,
                thirst: game.survival.thirst,
                temperature: game.survival.temperature,
            },
            world: {
                dayCount: game.world.dayCount,
                currentHour: game.world.currentHour,
                worldType: game.world.worldType || 'normal',
                returnPosition: game.world.returnPosition ? { x: game.world.returnPosition.x, y: game.world.returnPosition.y, z: game.world.returnPosition.z } : null,
            },
            buildings: _serializeBuildings(normalBldgs),
            stoneBuildings: _serializeBuildings(stoneBldgs),
            goldBuildings: _serializeBuildings(goldBldgs),
            achievements: game.achievements || {},
            achievementProgress: game.achievementProgress || {},
        };
        try {
            localStorage.setItem(getSlotSaveKey(username, slot), JSON.stringify(data));
        } catch (e) {
            console.error('Slot save failed:', e);
        }
    }

In [ ]:
// ===== SaveSystem.loadSlot =====
function loadSlot(username, slot) {
        try {
            const raw = localStorage.getItem(getSlotSaveKey(username, slot));
            if (!raw) return null;
            return JSON.parse(raw);
        } catch { return null; }
    }

In [ ]:
// ===== SaveSystem.hasSlotSave =====
function hasSlotSave(username, slot) {
        return localStorage.getItem(getSlotSaveKey(username, slot)) !== null;
    }

In [ ]:
// ===== SaveSystem.deleteSlotSave =====
function deleteSlotSave(username, slot) {
        localStorage.removeItem(getSlotSaveKey(username, slot));
    }

In [ ]:
// ===== SaveSystem.getSlotInfo =====
function getSlotInfo(username, slot) {
        const data = loadSlot(username, slot);
        if (!data) return null;
        return {
            day: data.world?.dayCount || 1,
            timestamp: data.timestamp || 0,
            health: data.survival?.health ?? 100,
        };
    }

In [ ]:
// ===== SaveSystem.getUsedSlotCount =====
function getUsedSlotCount(username) {
        let count = 0;
        for (let i = 1; i <= MAX_SLOTS; i++) {
            if (hasSlotSave(username, i)) count++;
        }
        return count;
    }

In [ ]:
// ===== SaveSystem.applySlotSave =====
function applySlotSave(username, slot, game) {
        const data = loadSlot(username, slot);
        if (!data) return false;
        return _applyData(data, game);
    }

In [ ]:
// ===== SaveSystem.getMpSaveKey =====
function getMpSaveKey(username, roomId) {
        return 'ws_mpsave_' + (roomId || 'unknown') + '_' + username;
    }

In [ ]:
// ===== SaveSystem._serializeBuildings =====
function _serializeBuildings(buildings) {
        return buildings.map(b => ({
            type: b.type,
            x: b.mesh.position.x,
            y: b.mesh.position.y,
            z: b.mesh.position.z,
            ry: b.mesh.rotation.y,
            minerTimer: b.minerTimer || 0,
            minerStorage: b.minerStorage || [],
            minerTools: (b.minerTools || []).map(tool => tool ? { id: tool.id, amount: tool.amount || 1 } : null),
            chestStorage: b.chestStorage || [],
            monumentName: b.monumentName || null,
            mysteryBoxStorage: b.mysteryBoxStorage || null,
            goldAltarStorage: b.goldAltarStorage || null,
            forgeStorage: b.forgeStorage || null,
        }));
    }

In [ ]:
// ===== SaveSystem.saveMultiplayer =====
function saveMultiplayer(username, roomId, game) {
        if (!username || !game) return;
        const normalBldgs = game.world.worldType === 'normal' ? game.world.buildings : (game.world._normalBuildings || []);
        const stoneBldgs = game.world.worldType === 'stone' ? game.world.buildings : (game.world._stoneBuildings || []);
        const goldBldgs = game.world.worldType === 'gold' ? game.world.buildings : (game.world._goldBuildings || []);
        const stoneCaveEnt = (game.world.worldType === 'stone')
            ? (game.world.getCaveEntrances ? game.world.getCaveEntrances() : (game.world._stoneCaveEntrances || []))
            : (game.world._stoneCaveEntrances || []);
        const data = {
            version: 1,
            timestamp: Date.now(),
            player: {
                x: game.player.position.x,
                y: game.player.position.y,
                z: game.player.position.z,
                yaw: game.player.yaw,
                pitch: game.player.pitch,
            },
            inventory: game.inventory.items.map(item => item ? { id: item.id, amount: item.amount, charge: item.charge || 0, uses: item.uses || 0 } : null),
            equipment: { ...game.inventory.equipment },
            survival: {
                health: game.survival.health,
                hunger: game.survival.hunger,
                thirst: game.survival.thirst,
                temperature: game.survival.temperature,
            },
            world: {
                dayCount: game.world.dayCount,
                currentHour: game.world.currentHour,
                worldType: game.world.worldType || 'normal',
                returnPosition: game.world.returnPosition ? { x: game.world.returnPosition.x, y: game.world.returnPosition.y, z: game.world.returnPosition.z } : null,
            },
            buildings: _serializeBuildings(normalBldgs),
            stoneBuildings: _serializeBuildings(stoneBldgs),
            goldBuildings: _serializeBuildings(goldBldgs),
            stoneCaveEntrances: stoneCaveEnt,
            achievements: game.achievements || {},
            achievementProgress: game.achievementProgress || {},
        };
        try {
            localStorage.setItem(getMpSaveKey(username, roomId), JSON.stringify(data));
        } catch (e) {
            console.error('MP save failed:', e);
        }
    }

In [ ]:
// ===== SaveSystem.loadMultiplayer =====
function loadMultiplayer(username, roomId) {
        try {
            const raw = localStorage.getItem(getMpSaveKey(username, roomId));
            if (!raw) return null;
            return JSON.parse(raw);
        } catch { return null; }
    }

In [ ]:
// ===== SaveSystem.hasMpSave =====
function hasMpSave(username, roomId) {
        return localStorage.getItem(getMpSaveKey(username, roomId)) !== null;
    }

In [ ]:
// ===== SaveSystem.applyMpSave =====
function applyMpSave(username, roomId, game) {
        const data = loadMultiplayer(username, roomId);
        if (!data) return false;
        // 复用 applySave 的逻辑
        return _applyData(data, game);
    }

In [ ]:
// ===== SaveSystem._applyData =====
function _applyData(data, game) {
        // 玩家位置
        if (data.player) {
            game.player.position.set(data.player.x, data.player.y, data.player.z);
            game.player.yaw = data.player.yaw || 0;
            game.player.pitch = data.player.pitch || 0;
        }

        // 背包
        if (data.inventory) {
            for (let i = 0; i < game.inventory.size && i < data.inventory.length; i++) {
                const saved = data.inventory[i];
                game.inventory.items[i] = saved ? { id: saved.id, amount: saved.amount, charge: saved.charge || 0, uses: saved.uses || 0 } : null;
            }
            game.inventory.notify();
        }

        // 装备
        if (data.equipment) {
            game.inventory.equipment = { head: null, chest: null, legs: null, feet: null, talisman1: null, talisman2: null, talisman3: null, talisman4: null };
            for (const slot of ['head', 'chest', 'legs', 'feet', 'talisman1', 'talisman2', 'talisman3', 'talisman4']) {
                if (data.equipment[slot]) {
                    game.inventory.equipment[slot] = { id: data.equipment[slot].id, amount: 1, charge: data.equipment[slot].charge || 0 };
                }
            }
        }

        // 同步盔甲值和护身符
        if (game.survival) {
            game.survival.setArmor(game.inventory.getArmorValue());
            const talismans = ['talisman1', 'talisman2', 'talisman3', 'talisman4'].map(s => game.inventory.equipment[s]).filter(Boolean);
            game.survival.setTalismans(talismans);
        }

        // 生存属性
        if (data.survival) {
            game.survival.health = data.survival.health ?? 100;
            game.survival.hunger = data.survival.hunger ?? 100;
            game.survival.thirst = data.survival.thirst ?? 100;
            game.survival.temperature = data.survival.temperature ?? 37;
            game.survival.dead = false;
            game.survival.notify();
        }

        // 世界时间
        if (data.world) {
            game.world.dayCount = data.world.dayCount || 1;
            game.world.currentHour = data.world.currentHour || 8.0;
        }

        // 成就
        if (data.achievements) {
            game.achievements = data.achievements;
        }
        if (data.achievementProgress) {
            game.achievementProgress = data.achievementProgress;
        }

        // 恢复建筑：先清除现有建筑，再从存档恢复
        for (const b of game.world.buildings) {
            game.scene.remove(b.mesh);
        }
        game.world.buildings = [];
        game.world.campfires = [];

        if (data.buildings) {
            for (const b of data.buildings) {
                const pos = new THREE.Vector3(b.x, b.y, b.z);
                game.world.placeBuilding(b.type, pos, b.ry || 0);
                const placed = game.world.buildings[game.world.buildings.length - 1];
                if (b.minerTimer) placed.minerTimer = b.minerTimer;
                if (b.minerStorage) placed.minerStorage = b.minerStorage;
                if (b.minerTools) placed.minerTools = b.minerTools;
                if (b.chestStorage) placed.chestStorage = b.chestStorage;
                if (b.monumentName) placed.monumentName = b.monumentName;
                if (b.mysteryBoxStorage) placed.mysteryBoxStorage = b.mysteryBoxStorage;
                if (b.goldAltarStorage) placed.goldAltarStorage = b.goldAltarStorage;
                if (b.forgeStorage) placed.forgeStorage = b.forgeStorage;
            }
        }

        // 恢复石头世界建筑（存入_stoneBuildings，不加入场景）
        game.world._stoneBuildings = null;
        game.world._stoneCampfires = null;
        if (data.stoneBuildings && data.stoneBuildings.length > 0) {
            const normalBldgCount = game.world.buildings.length;
            const normalCfCount = game.world.campfires.length;
            for (const b of data.stoneBuildings) {
                const pos = new THREE.Vector3(b.x, b.y, b.z);
                game.world.placeBuilding(b.type, pos, b.ry || 0);
                const placed = game.world.buildings[game.world.buildings.length - 1];
                if (b.minerTimer) placed.minerTimer = b.minerTimer;
                if (b.minerStorage) placed.minerStorage = b.minerStorage;
                if (b.minerTools) placed.minerTools = b.minerTools;
                if (b.chestStorage) placed.chestStorage = b.chestStorage;
                if (b.monumentName) placed.monumentName = b.monumentName;
                if (b.mysteryBoxStorage) placed.mysteryBoxStorage = b.mysteryBoxStorage;
                if (b.goldAltarStorage) placed.goldAltarStorage = b.goldAltarStorage;
                if (b.forgeStorage) placed.forgeStorage = b.forgeStorage;
            }
            game.world._stoneBuildings = game.world.buildings.splice(normalBldgCount);
            game.world._stoneCampfires = game.world.campfires.splice(normalCfCount);
            for (const b of game.world._stoneBuildings) {
                game.scene.remove(b.mesh);
            }
        }

        // 恢复金维度建筑（存入_goldBuildings，不加入场景）
        game.world._goldBuildings = null;
        game.world._goldCampfires = null;
        if (data.goldBuildings && data.goldBuildings.length > 0) {
            const normalBldgCount2 = game.world.buildings.length;
            const normalCfCount2 = game.world.campfires.length;
            for (const b of data.goldBuildings) {
                const pos = new THREE.Vector3(b.x, b.y, b.z);
                game.world.placeBuilding(b.type, pos, b.ry || 0);
                const placed = game.world.buildings[game.world.buildings.length - 1];
                if (b.minerTimer) placed.minerTimer = b.minerTimer;
                if (b.minerStorage) placed.minerStorage = b.minerStorage;
                if (b.minerTools) placed.minerTools = b.minerTools;
                if (b.chestStorage) placed.chestStorage = b.chestStorage;
                if (b.monumentName) placed.monumentName = b.monumentName;
                if (b.mysteryBoxStorage) placed.mysteryBoxStorage = b.mysteryBoxStorage;
                if (b.goldAltarStorage) placed.goldAltarStorage = b.goldAltarStorage;
                if (b.forgeStorage) placed.forgeStorage = b.forgeStorage;
            }
            game.world._goldBuildings = game.world.buildings.splice(normalBldgCount2);
            game.world._goldCampfires = game.world.campfires.splice(normalCfCount2);
            for (const b of game.world._goldBuildings) {
                game.scene.remove(b.mesh);
            }
        }

        // 恢复石维度矿洞入口位置（供后续进入石维度或切换时使用）
        if (data.stoneCaveEntrances && Array.isArray(data.stoneCaveEntrances)) {
            game.world._stoneCaveEntrances = data.stoneCaveEntrances;
        } else {
            game.world._stoneCaveEntrances = game.world._stoneCaveEntrances || null;
        }

        // 恢复维度：如果存档时在石头世界，切换到石头世界
        if (data.world && data.world.worldType === 'stone' && game.world.worldType !== 'stone') {
            const retPos = data.world.returnPosition
                ? new THREE.Vector3(data.world.returnPosition.x, data.world.returnPosition.y, data.world.returnPosition.z)
                : new THREE.Vector3(0, 0, 0);
            game.world.switchToWorld('stone', retPos);
            if (data.player) {
                const playerPos = new THREE.Vector3(data.player.x, data.player.y, data.player.z);
                game.world.updateChunks(playerPos);
            }
        }

        // 恢复维度：如果存档时在金维度，切换到金维度
        if (data.world && data.world.worldType === 'gold' && game.world.worldType !== 'gold') {
            const retPos = data.world.returnPosition
                ? new THREE.Vector3(data.world.returnPosition.x, data.world.returnPosition.y, data.world.returnPosition.z)
                : new THREE.Vector3(0, 0, 0);
            game.world.switchToWorld('gold', retPos);
            if (data.player) {
                const playerPos = new THREE.Vector3(data.player.x, data.player.y, data.player.z);
                game.world.updateChunks(playerPos);
            }
        }

        // 兜底：若当前已在石维度且有暂存入口，立即恢复（确保单人/多人切换后可见）
        if (game.world.worldType === 'stone' && game.world._stoneCaveEntrances && game.world.restoreCaveEntrances) {
            try { game.world.restoreCaveEntrances(game.world._stoneCaveEntrances); } catch (e) {}
        }

        return true;
    }

In [ ]:
// ===== SaveSystem.save =====
function save(username, game) {
        if (!username || !game) return;
        const normalBldgs = game.world.worldType === 'normal' ? game.world.buildings : (game.world._normalBuildings || []);
        const stoneBldgs = game.world.worldType === 'stone' ? game.world.buildings : (game.world._stoneBuildings || []);
        const goldBldgs = game.world.worldType === 'gold' ? game.world.buildings : (game.world._goldBuildings || []);
        const stoneCaveEnt = (game.world.worldType === 'stone')
            ? (game.world.getCaveEntrances ? game.world.getCaveEntrances() : (game.world._stoneCaveEntrances || []))
            : (game.world._stoneCaveEntrances || []);
        const data = {
            version: 1,
            timestamp: Date.now(),
            player: {
                x: game.player.position.x,
                y: game.player.position.y,
                z: game.player.position.z,
                yaw: game.player.yaw,
                pitch: game.player.pitch,
            },
            inventory: game.inventory.items.map(item => item ? { id: item.id, amount: item.amount, charge: item.charge || 0, uses: item.uses || 0 } : null),
            equipment: { ...game.inventory.equipment },
            survival: {
                health: game.survival.health,
                hunger: game.survival.hunger,
                thirst: game.survival.thirst,
                temperature: game.survival.temperature,
            },
            world: {
                dayCount: game.world.dayCount,
                currentHour: game.world.currentHour,
                worldType: game.world.worldType || 'normal',
                returnPosition: game.world.returnPosition ? { x: game.world.returnPosition.x, y: game.world.returnPosition.y, z: game.world.returnPosition.z } : null,
            },
            buildings: _serializeBuildings(normalBldgs),
            stoneBuildings: _serializeBuildings(stoneBldgs),
            goldBuildings: _serializeBuildings(goldBldgs),
            stoneCaveEntrances: stoneCaveEnt,
        };
        try {
            localStorage.setItem(getSaveKey(username), JSON.stringify(data));
        } catch (e) {
            console.error('Save failed:', e);
        }
    }

In [ ]:
// ===== SaveSystem.load =====
function load(username) {
        try {
            const raw = localStorage.getItem(getSaveKey(username));
            if (!raw) return null;
            return JSON.parse(raw);
        } catch { return null; }
    }

In [ ]:
// ===== SaveSystem.hasSave =====
function hasSave(username) {
        return localStorage.getItem(getSaveKey(username)) !== null;
    }

In [ ]:
// ===== SaveSystem.getSaveInfo =====
function getSaveInfo(username) {
        const data = load(username);
        if (!data) return null;
        return {
            day: data.world?.dayCount || 1,
            timestamp: data.timestamp || 0,
        };
    }

In [ ]:
// ===== SaveSystem.applySave =====
function applySave(username, game) {
        const data = load(username);
        if (!data) return false;
        return _applyData(data, game);
    }

In [ ]:
// ===== SaveSystem.getArcadeSaveKey =====
function getArcadeSaveKey(username) {
        return 'ws_arcade_save_' + username;
    }

In [ ]:
// ===== SaveSystem.saveArcade =====
function saveArcade(username, game) {
        if (!username || !game) return;
        const data = {
            version: 1,
            timestamp: Date.now(),
            berryUpgrades: game.berryUpgrades || { wood_sword: 0, wood_pickaxe: 0, efficiency: 1, collectCount: 0, randomizerCount: 0 },
            inventory: game.inventory.items.map(item => item ? { id: item.id, amount: item.amount, charge: item.charge || 0, uses: item.uses || 0 } : null),
            arcadeZombie: game.arcadeZombie || { active: false, score: 0, leaderboard: [] },
        };
        try {
            localStorage.setItem(getArcadeSaveKey(username), JSON.stringify(data));
        } catch (e) {
            console.error('Arcade save failed:', e);
        }
    }

In [ ]:
// ===== SaveSystem.hasArcadeSave =====
function hasArcadeSave(username) {
        return localStorage.getItem(getArcadeSaveKey(username)) !== null;
    }

In [ ]:
// ===== SaveSystem.loadArcadeSave =====
function loadArcadeSave(username) {
        try {
            const raw = localStorage.getItem(getArcadeSaveKey(username));
            if (!raw) return null;
            return JSON.parse(raw);
        } catch { return null; }
    }

In [ ]:
// ===== SaveSystem.applyArcadeSave =====
function applyArcadeSave(username, game) {
        const data = loadArcadeSave(username);
        if (!data) return false;
        if (data.berryUpgrades) {
            game.berryUpgrades = { ...data.berryUpgrades };
        }
        if (data.inventory) {
            const size = game.inventory.size || game.inventory.items.length;
            for (let i = 0; i < size && i < data.inventory.length; i++) {
                const saved = data.inventory[i];
                game.inventory.items[i] = saved ? { id: saved.id, amount: saved.amount, charge: saved.charge || 0, uses: saved.uses || 0 } : null;
            }
            game.inventory.notify();
        }
        if (data.arcadeZombie) {
            game.arcadeZombie = { ...data.arcadeZombie };
        }
        return true;
    }